# Session 13 — Coupling QOT = a Semidefinite Program

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

S13 promises and delivers a concrete computational definition of quantum optimal
transport. We:

1. Lift the **Kantorovich LP** to an **operator-level SDP** by replacing the transport
   plan $P \in T(a, b)$ with a bipartite density matrix $\rho_{AB}$ with prescribed
   partial-trace marginals.
2. Solve the SDP with **cvxpy**.
3. Verify the **diagonal-collapse principle** numerically (S12 promise).
4. Watch the SDP **distinguish $|+\rangle\langle+|$ from $I/2$** — exactly the gap
   classical OT could not see.
5. Validate against the closed-form pure-state identity $W_2^2 = 1 - |\langle\psi|\phi\rangle|^2$.

## 0. What you'll be able to do

- State the **quantum OT SDP** with marginal partial-trace constraints.
- Solve it in cvxpy with ~5 lines of code.
- Run the **diagonal-collapse check**: for commuting $\rho_A, \rho_B$, the SDP returns the classical $W_2^2$ on diagonals.
- Run the **$|+\rangle\langle+|$ vs $I/2$ punchline**: the SDP returns a *strictly positive* value, exposing what classical OT misses.
- Validate the **pure-state identity** $W_2^2(|\psi\rangle\langle\psi|, |\phi\rangle\langle\phi|) = 1 - |\langle\psi|\phi\rangle|^2$.
- Inspect the **optimal coupling** $\rho_{AB}^\star$ as a heatmap.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot
import cvxpy as cp

from qot_course import viz
from qot_course.quantum.composite import partial_trace
from qot_course.quantum.density import density_matrix, maximally_mixed
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, qubit_state
from qot_course.quantum_ot.sdp import (
    swap_matrix, swap_cost, quadratic_position_cost,
    quantum_ot_sdp, quantum_wasserstein_squared_swap,
)

np.random.seed(42)
viz.use_course_style()

## 1. From LP to SDP — the operator-level dictionary

The structural lift from S8 to S13 is a direct one-line correspondence:

| Classical (S8) | Quantum (S13) |
|---|---|
| transport plan $P \in \mathbb{R}_{\ge 0}^{n \times m}$ | bipartite density matrix $\rho_{AB} \succeq 0$ on $\mathcal{H}_A \otimes \mathcal{H}_B$ |
| marginals $P\,\mathbf{1} = a$, $P^\top\!\mathbf{1} = b$ | partial traces $\mathrm{tr}_B(\rho_{AB}) = \rho_A$, $\mathrm{tr}_A(\rho_{AB}) = \rho_B$ |
| cost dot product $\langle C, P\rangle$ | operator trace $\mathrm{tr}(C\,\rho_{AB})$ |
| **linear** program | **semidefinite** program (PSD cone replaces orthant) |

Putting it together — the **quantum optimal-transport SDP** (De Palma & Trevisan,
2021; Cole, Lostaglio, Verma, Wilde, 2023):

$$
\boxed{\;\mathrm{QOT}(\rho_A, \rho_B) \,=\,
   \min_{\rho_{AB} \succeq 0}\ \mathrm{tr}(C\,\rho_{AB})
   \quad \text{s.t.} \quad
   \mathrm{tr}_B(\rho_{AB}) = \rho_A,\ \ \mathrm{tr}_A(\rho_{AB}) = \rho_B.\;}
$$

A linear objective on a Hermitian PSD variable, with linear partial-trace constraints
— **cvxpy supports it directly**.

## 2. The 5-line cvxpy implementation

That's it — the entire algorithm:

```python
rho_ab = cp.Variable((d_a * d_b, d_a * d_b), hermitian=True)
constraints = [
    rho_ab >> 0,
    cp.partial_trace(rho_ab, (d_a, d_b), axis=1) == rho_a,
    cp.partial_trace(rho_ab, (d_a, d_b), axis=0) == rho_b,
]
problem = cp.Problem(cp.Minimize(cp.real(cp.trace(C @ rho_ab))), constraints)
problem.solve(solver="CLARABEL")
```

The PSD cone constraint replaces the orthant constraint of the LP — and that single
change captures every "quantum" effect we need. Two natural cost operators appear in
the literature:

- **SWAP cost** $C = I - \mathrm{SWAP}$ (Cole et al., 2023). Returns $1 - |\langle\psi|\phi\rangle|^2$ on pure states.
- **Quadratic position cost** $C = (X_A - X_B)^2$ with $X = \mathrm{diag}(0, 1, \dots)$. Reduces to the classical $W_2^2$ cost matrix $(i - j)^2$ on diagonal states (the diagonal-collapse principle).

We use both below.

## 3. Sanity: identical states return zero

The most basic test. For $\rho_A = \rho_B = \rho$, the diagonal coupling
$\rho_{AB} = \sum_k \lambda_k |k\rangle\langle k| \otimes |k\rangle\langle k|$
(in the eigenbasis of $\rho$) puts all mass on the diagonal of the cost operator. For
any cost that vanishes there ($I - \mathrm{SWAP}$, $(X_A - X_B)^2$), the SDP returns 0.

In [ ]:
rho = density_matrix(KET_PLUS)
value_swap, _ = quantum_ot_sdp(rho, rho, swap_cost(2))
value_pos, _  = quantum_ot_sdp(rho, rho, quadratic_position_cost([0.0, 1.0]))
print(f"QOT(|+><+|, |+><+|) with SWAP cost          = {value_swap:.6e}")
print(f"QOT(|+><+|, |+><+|) with quadratic position = {value_pos:.6e}")

**Read the output.** Both are zero to solver precision (~$10^{-9}$). Identical states
admit a perfect coupling — no transport cost. *Consistency check passed.*

## 4. The diagonal-collapse principle — the consistency check

S12 promised: for **commuting** $\rho_A, \rho_B$ (both diagonal in the same basis), the
quantum OT SDP must return the **classical $W_2^2$** on their diagonals. We finally
verify this numerically.

In [ ]:
# Two diagonal density matrices in the Z basis (commuting).
p_vec = np.array([0.6, 0.4])
q_vec = np.array([0.1, 0.9])
rho_a_diag = np.diag(p_vec).astype(complex)
rho_b_diag = np.diag(q_vec).astype(complex)

# Quantum SDP with the quadratic position cost (positions = {0, 1}).
positions = np.array([0.0, 1.0])
qot_value, plan = quantum_ot_sdp(
    rho_a_diag, rho_b_diag, quadratic_position_cost(positions)
)

# Classical W_2^2 with cost C[i, j] = (i - j)^2 on the same diagonals.
classical_cost = (positions.reshape(-1, 1) - positions.reshape(1, -1)) ** 2
w2_sq_classical = float(ot.emd2(p_vec, q_vec, classical_cost))

print(f"Quantum SDP value (QOT with quadratic cost) = {qot_value:.6f}")
print(f"Classical W_2^2  (LP on the Z-diagonals)    = {w2_sq_classical:.6f}")
print(f"diagonal-collapse match?                     {np.isclose(qot_value, w2_sq_classical, atol=1e-5)}")

**Read the output.** The two numbers agree. **The quantum SDP reduces to the classical
LP on commuting states** — Trevisan's consistency principle, confirmed by hand.

This is also why we *trust* the SDP: it agrees with everything we know from the
classical world, and only *adds new content* where classical OT was structurally blind.

## 5. The $|+\rangle\langle+|$ vs $I/2$ punchline — finally delivered

S12 set up the goal: classical $W$ on the Z-diagonals returns 0 for these two states,
but they are genuinely different. Now we run the SDP and watch it return a strictly
positive value.

In [ ]:
plus  = density_matrix(KET_PLUS)
mixed = maximally_mixed(2)

# Both have Z-diagonal (1/2, 1/2); classical OT on the diagonals returns 0.
diag_p = np.diag(plus).real.copy()
diag_m = np.diag(mixed).real.copy()
classical_zero = float(ot.emd2(
    np.ascontiguousarray(diag_p),
    np.ascontiguousarray(diag_m),
    (positions.reshape(-1, 1) - positions.reshape(1, -1)) ** 2,
))

# The SDP sees the difference.
qot_swap_value, plan_swap = quantum_ot_sdp(plus, mixed, swap_cost(2))
qot_quad_value, plan_quad = quantum_ot_sdp(plus, mixed, quadratic_position_cost([0.0, 1.0]))

print(f"Classical W_2^2 on Z-diagonals (both (1/2, 1/2)) = {classical_zero:.6f}   ← cannot tell them apart")
print(f"Quantum SDP, SWAP cost                            = {qot_swap_value:.6f}")
print(f"Quantum SDP, quadratic position cost              = {qot_quad_value:.6f}")
print()
print(f"both strictly positive?  {qot_swap_value > 1e-3 and qot_quad_value > 1e-3}")

**Read the output.** Classical OT on the diagonals: $0$. Quantum SDP: strictly positive,
both for the SWAP cost and the quadratic position cost. **The promise of M4 is
delivered**: the SDP sees what classical OT was blind to.

This is not just a sanity check — it is the formal *justification* for the entire
movement. We now have a single object (the SDP) that:

- agrees with classical OT when both states are diagonal in a common basis,
- distinguishes states that share a diagonal but differ in coherences or spectrum,
- is solvable as a standard SDP with off-the-shelf convex-optimisation tooling.

## 6. Pure-state validation — closed-form identity

For two pure states $\rho_A = |\psi\rangle\langle\psi|$ and
$\rho_B = |\phi\rangle\langle\phi|$, the unique coupling is the product
$\rho_{AB} = |\psi\rangle\langle\psi| \otimes |\phi\rangle\langle\phi|$ (because pure
marginals leave no slack). With the SWAP cost,

$$ \mathrm{tr}\bigl((I - \mathrm{SWAP})(|\psi\rangle\langle\psi| \otimes |\phi\rangle\langle\phi|)\bigr)
   = 1 - |\langle\psi|\phi\rangle|^2. $$

Let's verify the SDP returns this number on random qubit pairs.

In [ ]:
rng = np.random.default_rng(7)
print(f"{'|<psi|phi>|^2':>14s}  {'expected QOT':>14s}  {'SDP QOT':>14s}  {'match?':>8s}")
print("-" * 56)
for _ in range(5):
    psi = rng.normal(size=2) + 1j * rng.normal(size=2)
    psi /= np.linalg.norm(psi)
    phi = rng.normal(size=2) + 1j * rng.normal(size=2)
    phi /= np.linalg.norm(phi)
    overlap_sq = abs(np.vdot(psi, phi)) ** 2
    expected = 1.0 - overlap_sq
    actual = quantum_wasserstein_squared_swap(density_matrix(psi), density_matrix(phi))
    print(f"{overlap_sq:>14.4f}  {expected:>14.4f}  {actual:>14.4f}  "
          f"{str(np.isclose(actual, expected, atol=1e-3)):>8s}")

**Read the table.** Five random qubit pairs; the SDP returns $1 - |\langle\psi|\phi\rangle|^2$
to within solver precision (~$10^{-4}$). The closed-form identity is confirmed by the
solver — and conversely, the solver is confirmed by the closed form. This is exactly
the "trust through validation" discipline the old demo lacked.

## 7. The optimal coupling — see it

The SDP returns not just a *number* but the full **optimal coupling** $\rho_{AB}^\star$
— a $4 \times 4$ Hermitian PSD matrix for two qubits. Let's look at it for the
$|+\rangle\langle+|$ vs $I/2$ case.

In [ ]:
qot_value, optimal_plan = quantum_ot_sdp(plus, mixed, quadratic_position_cost([0.0, 1.0]))

# Sanity-check marginals.
rho_a_back = partial_trace(optimal_plan, keep=[0], dims=[2, 2])
rho_b_back = partial_trace(optimal_plan, keep=[1], dims=[2, 2])
print("marginal A recovered (should equal |+><+|):")
print(np.round(rho_a_back, 4))
print()
print("marginal B recovered (should equal I/2):")
print(np.round(rho_b_back, 4))
print()
print(f"trace of plan = {np.real(np.trace(optimal_plan)):.6f}")
print(f"minimum eigenvalue (PSD ⇒ >= 0): {float(np.min(np.linalg.eigvalsh(0.5 * (optimal_plan + optimal_plan.conj().T)))):.2e}")

# Plot real and imaginary parts of the optimal coupling.
viz.plot_density_matrix(optimal_plan, title=r"Optimal coupling $\rho^\star_{AB}$ for $|+\rangle\langle+|$  vs  $I/2$")
plt.show()

**Read the figure and the output.** The recovered marginals match $|+\rangle\langle+|$
and $I/2$ to numerical precision. The plan itself is a genuine bipartite density matrix
on $\mathcal{H}_A \otimes \mathcal{H}_B$, with a non-trivial structure that *no
classical coupling* could capture — the off-diagonal coherences of $|+\rangle\langle+|$
appear in the plan.

This is the operator-level analogue of the flow visualizations in S8 (sources and
targets connected by mass-weighted arrows). Here the "arrows" are operator
correlations between the two subsystems.

## 8. Dictionary update — the explicit quantum OT row

| Classical | Quantum |
|-----------|---------|
| transport plan $P \in T(a, b)$ | **quantum coupling $\rho_{AB} \succeq 0$ with partial-trace marginals** |
| Kantorovich LP $\min \langle C, P\rangle$ | **quantum OT SDP $\min \mathrm{tr}(C\,\rho_{AB})$** |
| transportation polytope $T(a, b)$ | **set of quantum couplings** (a *spectrahedron*) |
| $W_p$ Wasserstein distance | **$W_2^2 = \mathrm{QOT}$ for chosen cost** $C$ |
| diagonal-collapse principle ↔ classical LP | **diagonal-collapse principle ↔ same classical LP** (verified §4) |
| Sinkhorn (S10) | **quantum Sinkhorn** *(S14)* |

## 9. Your turn

1. **Two non-commuting same-diagonal states.** Recall $\rho_X = |+\rangle\langle+|$ and
   $\rho_Y = |+i\rangle\langle+i|$ from S12. Compute their quantum-Wasserstein-squared
   with the SWAP cost; compare to $1 - |\langle\psi_X|\psi_Y\rangle|^2$.
2. **Larger qubits.** Build a two-qubit example: $\rho_A = $ the Bell state's reduced
   state (= $I/2$), $\rho_B = $ a partial-trace of a noisy Bell. Solve the SDP with the
   quadratic position cost on 2-qubit positions $X = \mathrm{diag}(0, 1, 2, 3)$. Does
   the value behave smoothly as you sweep the noise?
3. **Convexity check.** Pick three density matrices $\rho, \sigma, \tau$ and verify
   numerically the **convexity** of $\mathrm{QOT}(\cdot, \tau)$:
   $\mathrm{QOT}\bigl((1{-}t)\rho + t\sigma,\,\tau\bigr) \le (1{-}t)\mathrm{QOT}(\rho, \tau) + t\,\mathrm{QOT}(\sigma, \tau)$
   for $t \in [0, 1]$.

## Conclusions & references

- **Quantum OT is a semidefinite program.** Five lines of cvxpy. The PSD cone replaces
  the LP orthant; the rest of the LP structure transfers verbatim.
- **The diagonal-collapse consistency principle holds numerically.** Commuting states
  give the classical answer.
- **The $|+\rangle\langle+|$ vs $I/2$ gap is resolved.** Classical OT on diagonals:
  $0$. Quantum SDP: strictly positive. The promise of M4 is delivered.
- **Validation against closed forms works**: pure-state QOT with SWAP cost equals
  $1 - |\langle\psi|\phi\rangle|^2$ to solver precision.
- **Next (S14 — Quantum Sinkhorn):** add the von-Neumann-entropy regulariser, derive
  the matrix-exponential analogue of S10's Sinkhorn iteration, and watch Amari's bridge
  (KL projection $\to$ Umegaki projection) survive the quantum lift.

**References:** G. De Palma, D. Trevisan, "Quantum optimal transport with quantum
channels", Ann. Henri Poincaré 22, 3199 (2021); S. Cole, M. Lostaglio, K. Verma,
M. M. Wilde, "Quantum Wasserstein distance based on an optimization over separable
states", IEEE Trans. Inf. Theory (2023); D. Trevisan, "Optimal transport methods for
quantum systems", arXiv:2202.02091 (2022); S. P. Boyd, L. Vandenberghe, *Convex
Optimization* (Cambridge, 2004), ch. 4 & 11 (SDPs); cvxpy docs:
https://www.cvxpy.org/. Previous: `notebooks/04_QuantumOptimalTransport/01_why_qot.ipynb`.